In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
RAIZ = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, RAIZ)
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from src.comun.rutas import DATA_GENERADO, MODELOS
M = json.load(open(os.path.join(MODELOS, "metricas.json"), encoding="utf-8")) if os.path.isfile(os.path.join(MODELOS, "metricas.json")) else {}
for f in os.listdir(MODELOS):
    if f.startswith("metricas_") and f.endswith(".json") and not f.startswith(("metricas_rag", "metricas_agente")):
        M[f[len("metricas_"):-5]] = json.load(open(os.path.join(MODELOS, f), encoding="utf-8"))
def tabla(d, dec=3):
    return pd.DataFrame(d).T.round(dec) if isinstance(d, dict) else pd.DataFrame(d)


# 01. Calidad de los datos sintéticos

Qué se generó, con qué semilla, cómo se evitó el sesgo de muestreo y qué controles pasan. Todo sale de `data/QUALITY.md` y de las tablas.

In [2]:
meta = json.load(open(os.path.join(DATA_GENERADO, 'meta.json'), encoding='utf-8')); meta

{'semilla': 2026,
 'clientes': 500,
 'inicio': '2019-01-01',
 'fin': '2025-12-31',
 'lote': 50,
 'movimientos': 3264148,
 'segundos': 39.9,
 'python': '3.13.2',
 'numpy': '2.5.4',
 'pandas': '3.1.1',
 'huella_parametros': {'generador.yaml': 'f532ab0d7d20e4f4',
  'eventos.yaml': '90b8415d9f9d66fa'},
 'incluye_prospectivos': False,
 'fuente_perfiles': 'parametrica'}

In [3]:
from src.generador.reporte import calcular
r = calcular(DATA_GENERADO)
print({k: ('OK' if v else 'FALLA') for k, v in r['controles'].items()})

{'Ningun estrato vacio': 'OK', 'Pesos muestrales entre 0,5 y 2': 'OK', 'transaction_id unico salvo duplicados exactos etiquetados': 'OK', 'Montos en soles positivos donde existen': 'OK', 'Fechas dentro de la ventana': 'OK', 'Mas dias en rojo en independientes que en dependientes formales': 'OK', 'Efecto del confinamiento: el ingreso del independiente de comercio cae a menos de 0,7 del de feb-2020': 'OK', 'Ingreso medio ponderado de 2025 dentro de +-35 % del ingreso urbano INEI': 'OK', 'Irregularidad: CV del ingreso por proyectos mayor que el del formal': 'OK', 'Saldo del ultimo movimiento coincide con el saldo diario (mediana de la diferencia < 1 sol)': 'OK', 'Sentimiento dentro de 5 puntos del reparto objetivo': 'OK', 'Faltantes dentro del 50 % relativo de lo configurado': 'OK', 'El mensaje negativo coincide con el saldo en rojo mas que el neutro': 'OK'}


## Estratos y pesos muestrales
Cada estrato (arquetipo × ahorro × gasto) tiene un piso; el peso restituye la proporción objetivo.

In [4]:
r['estratos'].round(3)

n   peso  participacion_muestra  participacion_objetivo
arquetipo       perfil_ahorro_real perfil_gasto_real                                                          
base_comision   ahorra             gasta_mucho        12  0.984                  0.024                   0.024
                                   gasta_poco         14  1.031                  0.028                   0.029
                no_ahorra          gasta_mucho        22  0.997                  0.044                   0.044
                                   gasta_poco         27  0.993                  0.054                   0.054
formal_quincena ahorra             gasta_mucho        23  1.027                  0.046                   0.047
                                   gasta_poco         44  0.997                  0.088                   0.088
                no_ahorra          gasta_mucho        29  0.996                  0.058                   0.058
                                   gasta_poco         54  0.993                  0.108                   0.107
independiente   ahorra             gasta_mucho        20  1.000                  0.040                   0.040
                                   gasta_poco         30  1.000                  0.060                   0.060
                no_ahorra          gasta_mucho        60  1.000                  0.120                   0.120
                                   gasta_poco         90  1.000                  0.180                   0.180
mixto           ahorra             gasta_mucho        10  0.900                  0.020                   0.018
                                   gasta_poco         13  1.038                  0.026                   0.027
                no_ahorra          gasta_mucho        21  1.000                  0.042                   0.042
                                   gasta_poco         31  1.016                  0.062                   0.063

## Ingreso por segmento y efecto del confinamiento

In [5]:
print(r['ingreso_2025'].round(0).to_dict()); print('abr-2020 / feb-2020:', r['covid'].round(2).to_dict()); print('ponderado 2025:', round(r['ingreso_2025_ponderado']), '| INEI urbano 2093,7')

{'base_comision': 2409.0, 'formal_quincena': 3023.0, 'independiente_diario': 1405.0, 'independiente_proyectos': 2508.0, 'mixto': 1928.0}
abr-2020 / feb-2020: {'base_comision': 0.6600000262260437, 'formal_quincena': 0.9100000262260437, 'independiente_diario': 0.38999998569488525, 'independiente_proyectos': 0.4000000059604645, 'mixto': 0.5099999904632568}
ponderado 2025: 2316 | INEI urbano 2093,7


In [6]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
r['rojo_anio'].plot(ax=ax[0], marker='o'); ax[0].set_title('Fracción de días con saldo negativo por año')
r['rojo_seg'].plot.bar(ax=ax[1]); ax[1].set_title('Por segmento'); plt.tight_layout(); plt.savefig(os.path.join(MODELOS, 'fig_rojo.png'), dpi=110); plt.show()